# Inventaire et premiers contrôles

Ce notebook vérifie les fichiers présents et affiche `info()` et `describe()` pour les données disponibles. Le jeu Fraud est signalé sans interrompre l'analyse s'il manque. Le CSV Lending Club est limité à un échantillon de 20 000 lignes pour éviter de charger plusieurs gigaoctets en mémoire.

Lancer Jupyter depuis la racine du dépôt. Les jeux ne sont jamais concaténés entre eux.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT_DIR = Path.cwd()
if not (ROOT_DIR / "data" / "raw").exists():
    ROOT_DIR = ROOT_DIR.parent

RAW_DIR = ROOT_DIR / "data" / "raw"
print(f"Données brutes : {RAW_DIR}")

Données brutes : /home/hawa-niass/Téléchargements/PLATEFORME INTELLIGENTE DE GESTION DES RISQUES BANCAIRES/data/raw


## Fonction de contrôle

Pour les CSV très volumineux, `nrows` permet de limiter la lecture. Chaque résultat décrit uniquement le fichier et, le cas échéant, l'échantillon indiqué.

In [2]:
def inspect_dataset(name, relative_path, *, separator=",", nrows=None, excel_header=0):
    """Charger un fichier disponible et afficher ses premiers contrôles."""
    path = RAW_DIR / relative_path
    if not path.is_file():
        print(f"\n{name} : fichier absent ({path})")
        return None

    if path.suffix.lower() == ".xls":
        frame = pd.read_excel(path, header=excel_header, engine="calamine")
    else:
        frame = pd.read_csv(path, sep=separator, nrows=nrows, low_memory=False)

    print(f"\n{name} — {path.name} — forme : {frame.shape}")
    frame.info(memory_usage="deep")
    print("\nStatistiques descriptives (aperçu des 12 premières variables) :")
    display(frame.describe(include="all").T.head(12))
    return frame

## Home Credit — jeu prioritaire

On commence par la table d'entraînement principale. Les agrégations multi-tables seront réalisées dans une étape dédiée, après l'examen des clés et de la qualité des données.

In [3]:
home_credit_train = inspect_dataset(
    "Home Credit (train)",
    "home_credit/application_train.csv",
)

if home_credit_train is not None:
    print("\nRépartition de TARGET :")
    display(home_credit_train["TARGET"].value_counts(dropna=False).rename("effectif"))
    display(home_credit_train["TARGET"].value_counts(normalize=True).rename("proportion"))


Home Credit (train) — application_train.csv — forme : (307511, 122)
<class 'pandas.DataFrame'>
RangeIndex: 307511 entries, 0 to 307510
Columns: 122 entries, SK_ID_CURR to AMT_REQ_CREDIT_BUREAU_YEAR
dtypes: float64(65), int64(41), str(16)
memory usage: 505.0 MB

Statistiques descriptives (aperçu des 12 premières variables) :


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
SK_ID_CURR,307511.0,NaN,NaN,NaN,278180.518577,102790.175348,100002.0,189145.5,278202.0,367142.5,456255.0
TARGET,307511.0,NaN,NaN,NaN,0.080729,0.272419,0.0,0.0,0.0,0.0,1.0
NAME_CONTRACT_TYPE,307511,2,Cash loans,278232,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CODE_GENDER,307511,3,F,202448,NaN,NaN,NaN,NaN,NaN,NaN,NaN
FLAG_OWN_CAR,307511,2,N,202924,NaN,NaN,NaN,NaN,NaN,NaN,NaN
FLAG_OWN_REALTY,307511,2,Y,213312,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CNT_CHILDREN,307511.0,NaN,NaN,NaN,0.417052,0.722121,0.0,0.0,0.0,1.0,19.0
AMT_INCOME_TOTAL,307511.0,NaN,NaN,NaN,168797.919297,237123.146279,25650.0,112500.0,147150.0,202500.0,117000000.0
AMT_CREDIT,307511.0,NaN,NaN,NaN,599025.999706,402490.776996,45000.0,270000.0,513531.0,808650.0,4050000.0
AMT_ANNUITY,307499.0,NaN,NaN,NaN,27108.573909,14493.737315,1615.5,16524.0,24903.0,34596.0,258025.5



Répartition de TARGET :


TARGET
0    282686
1     24825
Name: effectif, dtype: int64

TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64

## Jeux complémentaires disponibles

Les fichiers Lending Club ne sont lus que sur les 20 000 premières lignes de chaque CSV dans ce premier contrôle. Une analyse représentative complète nécessitera une lecture par blocs ou une sélection raisonnée des colonnes.

In [4]:
uci_credit_card = inspect_dataset(
    "UCI Credit Card Clients",
    "uci_credit_card/default of credit card clients.xls",
    excel_header=1,
)

bank_marketing = inspect_dataset(
    "Bank Marketing (full)",
    "bank_marketing/bank-full.csv",
    separator=";",
)

bank_additional = inspect_dataset(
    "Bank Marketing Additional (full)",
    "bank_marketing/bank-additional-full.csv",
    separator=";",
)

lending_accepted_sample = inspect_dataset(
    "Lending Club accepted (échantillon)",
    "lending_club/accepted_2007_to_2018Q4.csv",
    nrows=20_000,
)

lending_rejected_sample = inspect_dataset(
    "Lending Club rejected (échantillon)",
    "lending_club/rejected_2007_to_2018Q4.csv",
    nrows=20_000,
)

fraud_sample = inspect_dataset(
    "Credit Card Fraud",
    "fraud/creditcard.csv",
)

target_columns = [
    ("UCI Credit Card Clients", uci_credit_card, "default payment next month"),
    ("Bank Marketing", bank_marketing, "y"),
    ("Bank Marketing Additional", bank_additional, "y"),
    ("Credit Card Fraud", fraud_sample, "Class"),
]

for name, frame, target in target_columns:
    if frame is not None:
        print(f"\n{name} — répartition de {target} :")
        display(frame[target].value_counts(dropna=False).rename("effectif"))
        display(frame[target].value_counts(normalize=True, dropna=False).rename("proportion"))


UCI Credit Card Clients — default of credit card clients.xls — forme : (30000, 25)
<class 'pandas.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   ID                          30000 non-null  int64
 1   LIMIT_BAL                   30000 non-null  int64
 2   SEX                         30000 non-null  int64
 3   EDUCATION                   30000 non-null  int64
 4   MARRIAGE                    30000 non-null  int64
 5   AGE                         30000 non-null  int64
 6   PAY_0                       30000 non-null  int64
 7   PAY_2                       30000 non-null  int64
 8   PAY_3                       30000 non-null  int64
 9   PAY_4                       30000 non-null  int64
 10  PAY_5                       30000 non-null  int64
 11  PAY_6                       30000 non-null  int64
 12  BILL_AMT1                   30000 non-null 

,count,mean,std,min,25%,50%,75%,max
ID,30000.0,15000.500000,8660.398374,1.0,7500.75,15000.5,22500.25,30000.0
LIMIT_BAL,30000.0,167484.322667,129747.661567,10000.0,50000.00,140000.0,240000.00,1000000.0
SEX,30000.0,1.603733,0.489129,1.0,1.00,2.0,2.00,2.0
EDUCATION,30000.0,1.853133,0.790349,0.0,1.00,2.0,2.00,6.0
MARRIAGE,30000.0,1.551867,0.521970,0.0,1.00,2.0,2.00,3.0
AGE,30000.0,35.485500,9.217904,21.0,28.00,34.0,41.00,79.0
PAY_0,30000.0,-0.016700,1.123802,-2.0,-1.00,0.0,0.00,8.0
PAY_2,30000.0,-0.133767,1.197186,-2.0,-1.00,0.0,0.00,8.0
PAY_3,30000.0,-0.166200,1.196868,-2.0,-1.00,0.0,0.00,8.0
PAY_4,30000.0,-0.220667,1.169139,-2.0,-1.00,0.0,0.00,8.0



Bank Marketing (full) — bank-full.csv — forme : (45211, 17)
<class 'pandas.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   age        45211 non-null  int64
 1   job        45211 non-null  str  
 2   marital    45211 non-null  str  
 3   education  45211 non-null  str  
 4   default    45211 non-null  str  
 5   balance    45211 non-null  int64
 6   housing    45211 non-null  str  
 7   loan       45211 non-null  str  
 8   contact    45211 non-null  str  
 9   day        45211 non-null  int64
 10  month      45211 non-null  str  
 11  duration   45211 non-null  int64
 12  campaign   45211 non-null  int64
 13  pdays      45211 non-null  int64
 14  previous   45211 non-null  int64
 15  poutcome   45211 non-null  str  
 16  y          45211 non-null  str  
dtypes: int64(7), str(10)
memory usage: 25.7 MB

Statistiques descriptives (aperçu des 12 premières variables) :


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
age,45211.0,NaN,NaN,NaN,40.93621,10.618762,18.0,33.0,39.0,48.0,95.0
job,45211,12,blue-collar,9732,NaN,NaN,NaN,NaN,NaN,NaN,NaN
marital,45211,3,married,27214,NaN,NaN,NaN,NaN,NaN,NaN,NaN
education,45211,4,secondary,23202,NaN,NaN,NaN,NaN,NaN,NaN,NaN
default,45211,2,no,44396,NaN,NaN,NaN,NaN,NaN,NaN,NaN
balance,45211.0,NaN,NaN,NaN,1362.272058,3044.765829,-8019.0,72.0,448.0,1428.0,102127.0
housing,45211,2,yes,25130,NaN,NaN,NaN,NaN,NaN,NaN,NaN
loan,45211,2,no,37967,NaN,NaN,NaN,NaN,NaN,NaN,NaN
contact,45211,3,cellular,29285,NaN,NaN,NaN,NaN,NaN,NaN,NaN
day,45211.0,NaN,NaN,NaN,15.806419,8.322476,1.0,8.0,16.0,21.0,31.0



Bank Marketing Additional (full) — bank-additional-full.csv — forme : (41188, 21)
<class 'pandas.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  str    
 2   marital         41188 non-null  str    
 3   education       41188 non-null  str    
 4   default         41188 non-null  str    
 5   housing         41188 non-null  str    
 6   loan            41188 non-null  str    
 7   contact         41188 non-null  str    
 8   month           41188 non-null  str    
 9   day_of_week     41188 non-null  str    
 10  duration        41188 non-null  int64  
 11  campaign        41188 non-null  int64  
 12  pdays           41188 non-null  int64  
 13  previous        41188 non-null  int64  
 14  poutcome        41188 non-null  str    
 15  emp.var.rate    41188 non-null  float64
 16  cons

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
age,41188.0,NaN,NaN,NaN,40.02406,10.42125,17.0,32.0,38.0,47.0,98.0
job,41188,12,admin.,10422,NaN,NaN,NaN,NaN,NaN,NaN,NaN
marital,41188,4,married,24928,NaN,NaN,NaN,NaN,NaN,NaN,NaN
education,41188,8,university.degree,12168,NaN,NaN,NaN,NaN,NaN,NaN,NaN
default,41188,3,no,32588,NaN,NaN,NaN,NaN,NaN,NaN,NaN
housing,41188,3,yes,21576,NaN,NaN,NaN,NaN,NaN,NaN,NaN
loan,41188,3,no,33950,NaN,NaN,NaN,NaN,NaN,NaN,NaN
contact,41188,2,cellular,26144,NaN,NaN,NaN,NaN,NaN,NaN,NaN
month,41188,10,may,13769,NaN,NaN,NaN,NaN,NaN,NaN,NaN
day_of_week,41188,5,thu,8623,NaN,NaN,NaN,NaN,NaN,NaN,NaN


... (14 outputs left out)

y
no     0.887346
yes    0.112654
Name: proportion, dtype: float64


Credit Card Fraud — répartition de Class :


Class
0    284315
1       492
Name: effectif, dtype: int64

Class
0    0.998273
1    0.001727
Name: proportion, dtype: float64